# 🚛 Driver Recruit Environment — GRPO Training with TRL

Train an LLM to recruit truck drivers using TRL's GRPOTrainer.

The model controls every action in the episode via JSON tool calls:
CRM reads, messaging, stage updates, approval, and hiring.

**Environment**: [OpenEnv 0.2.1](https://github.com/meta-pytorch/OpenEnv) deployed on HF Spaces

**Model**: Qwen/Qwen2.5-1.5B-Instruct

**Algorithm**: TRL GRPOTrainer with multi-turn rollout_func

## 1. Install Dependencies

In [ ]:
!pip install -q "openenv-core[core]==0.2.1" trl transformers torch accelerate datasets vllm

## 2. Connect to the Environment

The recruiting environment is deployed on HF Spaces. Replace the URL below with your Space URL.

In [ ]:
import json
import random
import requests

# Replace with your HF Space URL
ENV_URL = "https://YOUR-USERNAME-recruitopenenv.hf.space"  # <-- CHANGE THIS

# Quick test
resp = requests.post(f"{ENV_URL}/reset", json={"seed": 42})
data = resp.json()
print(f"Driver: {data['observation']['driver_name']}")
print(f"Stage: {data['observation']['stage']}")
print("Environment connected!")

## 3. System Prompt & Helpers

The model outputs structured JSON tool calls to interact with the CRM, messaging, approval, and workflow tools.

In [ ]:
SYSTEM_PROMPT = """You are a truck driver recruiter using a CRM system. You only know the driver's name. You must discover their qualifications through conversation, record info in the CRM, get approval, and hire them.

You have 4 tools:

## crm
- read_candidate: Read the current CRM record
- update_stage: Advance pipeline (contacted \u2192 interested \u2192 approval_pending \u2192 offer_sent \u2192 hired)
- update_field: Record info (field + value)
- add_note: Add a free-text note

## messaging
- send_message: Send a message (topic: greeting, call, experience, home_time, pay, equipment, route, deal_breakers, availability, violations, medical_card, references, pitch, offer)
- read_reply: Read the driver's response

## approval
- request_approval: Request approval for a job (needs job_id)
- check_approval: Check approval status

## workflow
- wait: Advance time (needed for approval processing)

## Rules
- Must read CRM before messaging
- Must read_reply before sending another message
- Must request_approval and wait before sending offer
- Must follow stage order: lead \u2192 contacted \u2192 interested \u2192 approval_pending \u2192 offer_sent \u2192 hired

## Workflow
1. crm.read_candidate
2. messaging.send_message (greeting/call) \u2192 read_reply \u2192 update_stage(contacted)
3. messaging.send_message (screening topics) \u2192 read_reply \u2192 crm.update_field
4. crm.update_stage(interested)
5. approval.request_approval \u2192 workflow.wait \u2192 approval.check_approval
6. crm.update_stage(approval_pending)
7. messaging.send_message(offer) \u2192 read_reply
8. crm.update_stage(offer_sent) \u2192 crm.update_stage(hired)

Respond with ONLY JSON:
{\"tool\": \"crm\", \"action\": \"read_candidate\"}
{\"tool\": \"messaging\", \"action\": \"send_message\", \"topic\": \"experience\"}
{\"tool\": \"messaging\", \"action\": \"read_reply\"}
{\"tool\": \"crm\", \"action\": \"update_stage\", \"stage\": \"contacted\"}
{\"tool\": \"approval\", \"action\": \"request_approval\", \"job_id\": 2}
{\"tool\": \"workflow\", \"action\": \"wait\"}
{\"tool\": \"crm\", \"action\": \"update_stage\", \"stage\": \"hired\"}"""


def format_observation(obs):
    """Format observation into a user prompt for the LLM."""
    parts = [f"Driver: {obs['driver_name']}"]
    if obs.get('crm_summary'):
        parts.append(f"CRM:\n{obs['crm_summary']}")
    if obs.get('jobs_summary'):
        parts.append(f"Jobs:\n{obs['jobs_summary']}")
    if obs.get('discovered_info'):
        parts.append(f"Discovered:\n{obs['discovered_info']}")
    status = f"Stage: {obs['stage']}"
    if obs.get('pending_reply'):
        status += " | PENDING REPLY"
    parts.append(status)
    if obs.get('feedback'):
        parts.append(f"Result: {obs['feedback']}")
    return "\n".join(parts)


def parse_action(text):
    """Parse LLM JSON output into an action dict."""
    text = text.strip()
    if "```" in text:
        for part in text.split("```"):
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                text = part
                break
    try:
        data = json.loads(text)
        if isinstance(data, list):
            data = data[0] if data else {}
        if isinstance(data, dict) and "tool" in data and "action" in data:
            return data
    except (json.JSONDecodeError, KeyError, IndexError):
        pass
    # Fallback
    text_lower = text.lower()
    if "read_candidate" in text_lower:
        return {"tool": "crm", "action": "read_candidate"}
    if "read_reply" in text_lower:
        return {"tool": "messaging", "action": "read_reply"}
    if "wait" in text_lower:
        return {"tool": "workflow", "action": "wait"}
    return {"tool": "crm", "action": "read_candidate"}


def env_reset(seed=None):
    payload = {"seed": seed} if seed else {}
    return requests.post(f"{ENV_URL}/reset", json=payload).json()


def env_step(action_dict):
    return requests.post(f"{ENV_URL}/step", json=action_dict).json()


print("Helpers loaded!")

## 4. Run a Demo Episode

Watch the model run a full multi-turn recruiting episode with JSON tool calls.

In [ ]:
def run_demo_episode(seed=42):
    """Run one full episode with scripted actions."""
    state = env_reset(seed=seed)
    obs = state["observation"]
    total_reward = 0.0
    print(f"=== Driver: {obs['driver_name']} ===")

    actions = [
        {"tool": "crm", "action": "read_candidate"},
        {"tool": "messaging", "action": "send_message", "topic": "greeting"},
        {"tool": "messaging", "action": "read_reply"},
        {"tool": "crm", "action": "update_stage", "stage": "contacted"},
        {"tool": "messaging", "action": "send_message", "topic": "experience"},
        {"tool": "messaging", "action": "read_reply"},
        {"tool": "messaging", "action": "send_message", "topic": "deal_breakers"},
        {"tool": "messaging", "action": "read_reply"},
        {"tool": "messaging", "action": "send_message", "topic": "pay"},
        {"tool": "messaging", "action": "read_reply"},
        {"tool": "crm", "action": "update_stage", "stage": "interested"},
        {"tool": "approval", "action": "request_approval", "job_id": 0},
        {"tool": "workflow", "action": "wait"},
        {"tool": "approval", "action": "check_approval"},
        {"tool": "crm", "action": "update_stage", "stage": "approval_pending"},
        {"tool": "messaging", "action": "send_message", "topic": "offer", "job_id": 0},
        {"tool": "messaging", "action": "read_reply"},
        {"tool": "crm", "action": "update_stage", "stage": "offer_sent"},
        {"tool": "crm", "action": "update_stage", "stage": "hired"},
    ]

    for a in actions:
        if state.get("done"):
            break
        state = env_step(a)
        total_reward += state["reward"]
        obs = state["observation"]
        print(f"  {a['tool']}.{a['action']}: reward={state['reward']:.1f}, stage={obs['stage']}")

    print(f"\nFinal stage: {obs['stage']}")
    print(f"Total reward: {total_reward:.1f}")
    return total_reward

run_demo_episode()

## 5. TRL GRPO Training Setup

Uses TRL's GRPOTrainer with a custom `rollout_func` for multi-turn episodes.
Each episode runs the full recruiting pipeline with the model generating JSON tool calls.

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_COMPLETION_TOKENS = 1536
MAX_TURNS = 15

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

In [ ]:
def format_observation_compact(obs):
    """Compact observation for embedding in completion_ids."""
    parts = [f"Stage: {obs['stage']}"]
    if obs.get('pending_reply'):
        parts.append("PENDING REPLY")
    if obs.get('feedback'):
        parts.append(str(obs['feedback'])[:200])
    if obs.get('discovered_info'):
        parts.append(str(obs['discovered_info'])[:200])
    return "\n".join(parts)


def _build_chat_transition(tokenizer, obs_text):
    """Build chat-formatted transition tokens between turns."""
    im_start = tokenizer.convert_tokens_to_ids("<|im_start|>")
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    nl = tokenizer.encode("\n", add_special_tokens=False)
    user_tag = tokenizer.encode("user", add_special_tokens=False)
    asst_tag = tokenizer.encode("assistant", add_special_tokens=False)
    obs_ids = tokenizer.encode(obs_text, add_special_tokens=False)[:60]
    return (
        [im_end] + nl +
        [im_start] + user_tag + nl +
        obs_ids +
        [im_end] + nl +
        [im_start] + asst_tag + nl
    )


def rollout_once(trainer, tokenizer, system_prompt, max_turns=15):
    """Run one multi-turn episode with chat-formatted transitions."""
    seed = random.randint(0, 2**31 - 1)
    state = env_reset(seed=seed)
    obs = state["observation"]

    prompt_ids = []
    completion_ids = []
    logprobs = []
    env_mask_list = []
    total_reward = 0.0
    steps = 0

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": format_observation(obs)},
    ]

    while not state.get("done") and steps < max_turns:
        if len(completion_ids) > MAX_COMPLETION_TOKENS - 60:
            break

        current_prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        rollout_outputs = generate_rollout_completions(trainer, [current_prompt])[0]

        if steps == 0:
            prompt_ids = list(rollout_outputs["prompt_ids"])

        action_ids = list(rollout_outputs["completion_ids"])
        action_logprobs = list(rollout_outputs["logprobs"])

        completion_ids.extend(action_ids)
        logprobs.extend(action_logprobs)
        env_mask_list.extend([1] * len(action_ids))

        response = rollout_outputs.get("text") or tokenizer.decode(action_ids, skip_special_tokens=True)
        messages.append({"role": "assistant", "content": response})

        action_dict = parse_action(response)
        state = env_step(action_dict)
        obs = state["observation"]
        total_reward += state["reward"]
        steps += 1

        if not state.get("done"):
            obs_text = format_observation_compact(obs)
            transition_ids = _build_chat_transition(tokenizer, obs_text)
            completion_ids.extend(transition_ids)
            logprobs.extend([0.0] * len(transition_ids))
            env_mask_list.extend([0] * len(transition_ids))
            messages.append({"role": "user", "content": format_observation(obs)})

    completion_ids = completion_ids[:MAX_COMPLETION_TOKENS]
    logprobs = logprobs[:MAX_COMPLETION_TOKENS]
    env_mask_list = env_mask_list[:MAX_COMPLETION_TOKENS]

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "env_mask": env_mask_list,
        "env_reward": total_reward,
        "steps": steps,
        "final_stage": obs.get("stage", "unknown"),
    }


def rollout_func(prompts, trainer):
    """Multi-turn rollout: model controls every action in the episode."""
    all_prompt_ids = []
    all_completion_ids = []
    all_logprobs = []
    all_env_rewards = []
    all_env_mask = []

    for prompt_text in prompts:
        episode = rollout_once(trainer, tokenizer, SYSTEM_PROMPT)

        if episode["completion_ids"]:
            all_prompt_ids.append(episode["prompt_ids"])
            all_completion_ids.append(episode["completion_ids"])
            all_logprobs.append(episode["logprobs"])
            all_env_mask.append(episode["env_mask"])
        else:
            tok_ids = tokenizer.encode("wait", add_special_tokens=False)
            all_prompt_ids.append(episode["prompt_ids"] or tok_ids)
            all_completion_ids.append(tok_ids)
            all_logprobs.append([0.0] * len(tok_ids))
            all_env_mask.append([1] * len(tok_ids))

        all_env_rewards.append(episode["env_reward"])
        print(f"  Episode {len(all_env_rewards)}: reward={episode['env_reward']:.1f}, "
              f"steps={episode['steps']}, stage={episode['final_stage']}")

    mean_r = sum(all_env_rewards) / len(all_env_rewards)
    std_r = torch.tensor(all_env_rewards).std().item()
    print(f"Rollout done: {len(all_env_rewards)} episodes, mean_reward={mean_r:.2f}, std={std_r:.2f}")

    return {
        "prompt_ids": all_prompt_ids,
        "completion_ids": all_completion_ids,
        "logprobs": [[(lp,) for lp in seq] for seq in all_logprobs],
        "env_reward": all_env_rewards,
        "env_mask": all_env_mask,
    }


def reward_total(completions, **kwargs):
    """Extract environment rewards passed via rollout_func kwargs."""
    env_rewards = kwargs.get("env_reward", [])
    if env_rewards:
        return [float(r) for r in env_rewards]
    return [0.0] * len(completions)


print("Rollout functions defined!")

In [ ]:
# --- Build dataset of initial prompts ---
prompts = []
for i in range(16):
    state = env_reset()
    obs = state["observation"]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_observation(obs)},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    prompts.append(prompt_text)

dataset = Dataset.from_dict({"prompt": prompts})
print(f"Dataset: {len(dataset)} prompts")
print(f"Sample prompt length: {len(tokenizer.encode(prompts[0]))} tokens")

In [ ]:
# --- Configure and launch GRPOTrainer ---
grpo_config = GRPOConfig(
    output_dir="./recruit-grpo-output",
    use_vllm=True,
    vllm_mode="colocate",
    num_train_epochs=1,
    num_generations=4,
    max_completion_length=1536,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    temperature=0.7,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    report_to="none",
    run_name="recruit-grpo-colab",
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    processing_class=tokenizer,
    reward_funcs=[reward_total],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

print("=" * 50)
print(f"Training {MODEL_NAME} (TOOL-BASED MULTI-TURN)")
print(f"Environment: {ENV_URL}")
print(f"Episodes: {len(dataset)}")
print(f"Generations per prompt: 4")
print(f"Max completion tokens: 1536")
print("=" * 50)

## 6. Train

In [ ]:
trainer.train()
trainer.save_model("./recruit-grpo-output")
print("\nTraining complete! Model saved to ./recruit-grpo-output")

## 7. Evaluate the Trained Model

In [ ]:
from transformers import AutoModelForCausalLM
import torch.nn.functional as F

# Load trained model for evaluation
eval_model = AutoModelForCausalLM.from_pretrained(
    "./recruit-grpo-output",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
device = next(eval_model.parameters()).device

def eval_episode(model, seed=None):
    """Run one episode with the trained model generating actions."""
    if seed is None:
        seed = random.randint(0, 2**31 - 1)
    state = env_reset(seed=seed)
    obs = state["observation"]
    total_reward = 0.0

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_observation(obs)},
    ]

    for step in range(20):
        if state.get("done"):
            break
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=64, do_sample=True, temperature=0.7,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        response = tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True)
        messages.append({"role": "assistant", "content": response})

        action_dict = parse_action(response)
        state = env_step(action_dict)
        obs = state["observation"]
        total_reward += state["reward"]

        if not state.get("done"):
            messages.append({"role": "user", "content": format_observation(obs)})

    return {"reward": total_reward, "stage": obs.get("stage", "unknown"), "steps": step + 1}

print("=== Evaluating trained model ===")
results = []
for i in range(5):
    r = eval_episode(eval_model)
    results.append(r)
    print(f"  Episode {i+1}: reward={r['reward']:.1f}, stage={r['stage']}, steps={r['steps']}")

print(f"\nMean reward: {sum(r['reward'] for r in results) / len(results):.1f}")